In [89]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-1"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()


### ways to create a DF

In [2]:
spark.sql("use itv024128")

""


In [3]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,groceries,false
itv024128,groceries_ext,false
itv024128,groceries_ext_json,false
itv024128,groceries_json,false
itv024128,orders_ext,false


In [4]:
spark.sql("select * from itv024128.groceries").show(3)

+--------+--------+-------+----------+--------+
|order_id|location|   item|order_date|quantity|
+--------+--------+-------+----------+--------+
|      o1| Seattle|Bananas|01/01/2017|       7|
|      o2|    Kent| Apples|02/01/2017|      20|
|      o3|Bellevue|Flowers|02/01/2017|      10|
+--------+--------+-------+----------+--------+
only showing top 3 rows



## from a spark table select query

In [5]:
df1 = spark.sql("select * from itv024128.groceries")

In [6]:
df1.show(3)

+--------+--------+-------+----------+--------+
|order_id|location|   item|order_date|quantity|
+--------+--------+-------+----------+--------+
|      o1| Seattle|Bananas|01/01/2017|       7|
|      o2|    Kent| Apples|02/01/2017|      20|
|      o3|Bellevue|Flowers|02/01/2017|      10|
+--------+--------+-------+----------+--------+
only showing top 3 rows



## from spark.table func

In [7]:
df2 = spark.table("itv024128.groceries")

In [8]:
df2.show(3)

+--------+--------+-------+----------+--------+
|order_id|location|   item|order_date|quantity|
+--------+--------+-------+----------+--------+
|      o1| Seattle|Bananas|01/01/2017|       7|
|      o2|    Kent| Apples|02/01/2017|      20|
|      o3|Bellevue|Flowers|02/01/2017|      10|
+--------+--------+-------+----------+--------+
only showing top 3 rows



## from spark.range()

In [9]:
df3 = spark.range(10)

In [10]:
df3.show(5)

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
only showing top 5 rows



In [11]:
df3 = spark.range(1,10)

In [12]:
df3.show(5)

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
+---+
only showing top 5 rows



In [13]:
df3 = spark.range(2,12,2)

In [14]:
df3.show(5)

+---+
| id|
+---+
|  2|
|  4|
|  6|
|  8|
| 10|
+---+



## spark.createDataFrame command

In [15]:
list1 = (1,2,3,4,5,6,7,8,9,)

In [16]:
rdd1 = spark.sparkContext.parallelize(list1)

In [17]:
rdd1.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [18]:
rdd1.getNumPartitions()

2

In [19]:
##create DF from an RDD
# [itv024128@g02 ~]$ hadoop fs -head /public/trendytech/retail_db/orders/part-00000
# 1,2013-07-25 00:00:00.0,11599,CLOSED
# 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
# 3,2013-07-25 00:00:00.0,12111,COMPLETE
# 4,2013-07-25 00:00:00.0,8827,CLOSED
# 5,2013-07-25 00:00:00.0,11318,COMPLETE

In [20]:
rdd1 = spark.sparkContext.textFile("/public/trendytech/retail_db/orders/part-00000")

In [21]:
rdd1.take(3)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE']

In [22]:
rdd2 = rdd1.map(lambda x: (int(x.split(",")[0]),x.split(",")[1],int(x.split(",")[2]),x.split(",")[3]))

In [23]:
rdd2.take(3)

[(1, '2013-07-25 00:00:00.0', 11599, 'CLOSED'),
 (2, '2013-07-25 00:00:00.0', 256, 'PENDING_PAYMENT'),
 (3, '2013-07-25 00:00:00.0', 12111, 'COMPLETE')]

In [24]:
df_rdd0 = rdd2.toDF()

In [25]:
df_rdd0.show(3)

+---+--------------------+-----+---------------+
| _1|                  _2|   _3|             _4|
+---+--------------------+-----+---------------+
|  1|2013-07-25 00:00:...|11599|         CLOSED|
|  2|2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:...|12111|       COMPLETE|
+---+--------------------+-----+---------------+
only showing top 3 rows



In [26]:
schema_rdd = ("order_id","orderTS","cust_id","ord_stat") ## or schema_rdd = 'order_id long,orderTS string,cust_id long,ord_stat string'

In [27]:
df_rdd = rdd2.toDF(schema_rdd)  ## names the columns

In [28]:
df_rdd.show(4)

+--------+--------------------+-------+---------------+
|order_id|             orderTS|cust_id|       ord_stat|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
|       4|2013-07-25 00:00:...|   8827|         CLOSED|
+--------+--------------------+-------+---------------+
only showing top 4 rows



In [29]:
df_rdd.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- orderTS: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- ord_stat: string (nullable = true)



In [30]:
# another option is to use createDataFrame

In [31]:
schema_rdd1 = 'order_id long,orderTS string,cust_id long,ord_stat string'

In [32]:
df_rdd1 = spark.createDataFrame(rdd2,schema=schema_rdd1) ## enforces schema

In [33]:
df_rdd1.show(4)

+--------+--------------------+-------+---------------+
|order_id|             orderTS|cust_id|       ord_stat|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
|       4|2013-07-25 00:00:...|   8827|         CLOSED|
+--------+--------------------+-------+---------------+
only showing top 4 rows



In [34]:
df_rdd1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- orderTS: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- ord_stat: string (nullable = true)



In [35]:
schema1 = ("order_id","orderTS","cust_id","ord_stat") 

In [36]:
df_rdd2 = spark.createDataFrame(rdd2,schema1) ## no schema enforcement, just column names assigment

In [37]:
df_rdd2.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- orderTS: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- ord_stat: string (nullable = true)



In [38]:
# convert a local python list to DF

In [39]:
list2  =[(1,'07-25-2013',11599,'CLOSED'),
(2,'07-25-2013',256,'PENDING_PAYMENT'),
(3,'07-25-2013',12111,'COMPLETE')]

In [40]:
orders_df=spark.createDataFrame(list2)

In [41]:
orders_df.show()

+---+----------+-----+---------------+
| _1|        _2|   _3|             _4|
+---+----------+-----+---------------+
|  1|07-25-2013|11599|         CLOSED|
|  2|07-25-2013|  256|PENDING_PAYMENT|
|  3|07-25-2013|12111|       COMPLETE|
+---+----------+-----+---------------+



In [42]:
## schema infered

In [43]:
orders_df.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)
 |-- _3: long (nullable = true)
 |-- _4: string (nullable = true)



In [44]:
## assigning schema to created DF

In [45]:
schema1 = 'order_id long, order_date string , cust_id long, status string'

In [46]:
orders_df1=spark.createDataFrame(list2,schema1)

In [47]:
orders_df1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: string (nullable = true)



In [48]:
orders_df1.show()

+--------+----------+-------+---------------+
|order_id|order_date|cust_id|         status|
+--------+----------+-------+---------------+
|       1|07-25-2013|  11599|         CLOSED|
|       2|07-25-2013|    256|PENDING_PAYMENT|
|       3|07-25-2013|  12111|       COMPLETE|
+--------+----------+-------+---------------+



#### toDF  - assign new column names

In [49]:
orders_df2=spark.createDataFrame(list2).toDF('order_id' , 'order_date'  , 'cust_id' , 'status')

In [50]:
orders_df2.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- status: string (nullable = true)



In [51]:
orders_df2.show()

+--------+----------+-------+---------------+
|order_id|order_date|cust_id|         status|
+--------+----------+-------+---------------+
|       1|07-25-2013|  11599|         CLOSED|
|       2|07-25-2013|    256|PENDING_PAYMENT|
|       3|07-25-2013|  12111|       COMPLETE|
+--------+----------+-------+---------------+



In [52]:
# OR

In [53]:
col_names= ['order_id','ord_date','custid','status']

In [54]:
orders_df3=spark.createDataFrame(list2,col_names)

In [55]:
orders_df3.show()

+--------+----------+------+---------------+
|order_id|  ord_date|custid|         status|
+--------+----------+------+---------------+
|       1|07-25-2013| 11599|         CLOSED|
|       2|07-25-2013|   256|PENDING_PAYMENT|
|       3|07-25-2013| 12111|       COMPLETE|
+--------+----------+------+---------------+



In [56]:
orders_df3.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- ord_date: string (nullable = true)
 |-- custid: long (nullable = true)
 |-- status: string (nullable = true)



### Nested Schema

In [57]:
!hadoop fs -head /public/trendytech/datasets/customer_nested/part-00000-950ffc21-f8aa-4e00-8181-8ab726051097-c000.json

{"customer_id":1,"fullname":{"firstname":"sumit","lastname":"mittal"},"city":"bangalore"}


In [59]:
!hadoop fs -head /public/trendytech/datasets/customer_nested/part-00001-950ffc21-f8aa-4e00-8181-8ab726051097-c000.json

{"customer_id":2,"fullname":{"firstname":"ram","lastname":"kumar"},"city":"hyderabad"}
{"customer_id":3,"fullname":{"firstname":"vijay","lastname":"shankar"},"city":"pune"}


In [60]:
#Schema DDL

In [66]:
ddl_schema = "customer_id long , fullname struct<firstname:string, lastname:string>, city string"

In [67]:
df_json = spark.read.format("json").schema(ddl_schema).load("/public/trendytech/datasets/customer_nested/*")

In [68]:
df_json.show()

+-----------+----------------+---------+
|customer_id|        fullname|     city|
+-----------+----------------+---------+
|          2|    {ram, kumar}|hyderabad|
|          3|{vijay, shankar}|     pune|
|          1| {sumit, mittal}|bangalore|
+-----------+----------------+---------+



In [69]:
df_json.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- fullname: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- city: string (nullable = true)



In [70]:
from pyspark.sql.types import *

In [82]:
cust_schema = StructType ([
    StructField("customer_id",LongType()),
    StructField("fullname",StructType([StructField("firstname",StringType()),StructField("lastname",StringType())])),
    StructField("city",StringType())
])

In [83]:
df_json1 = spark.read.format("json").schema(cust_schema).load("/public/trendytech/datasets/customer_nested/*")

In [84]:
df_json1.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- fullname: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- city: string (nullable = true)



In [85]:
df_json1.show()

+-----------+----------------+---------+
|customer_id|        fullname|     city|
+-----------+----------------+---------+
|          2|    {ram, kumar}|hyderabad|
|          3|{vijay, shankar}|     pune|
|          1| {sumit, mittal}|bangalore|
+-----------+----------------+---------+



In [86]:
# nested list

In [87]:
nest_list = [
    (1,("fname1","lname1"),"city1"),
    (2,("fname2","lname2"),"city2"),
    (3,("fname3","lname3"),"city3"),
]

In [90]:
nest_list_df = spark.createDataFrame(nest_list,schema=cust_schema)

In [91]:
nest_list_df.show()

+-----------+----------------+-----+
|customer_id|        fullname| city|
+-----------+----------------+-----+
|          1|{fname1, lname1}|city1|
|          2|{fname2, lname2}|city2|
|          3|{fname3, lname3}|city3|
+-----------+----------------+-----+



In [92]:
nest_list_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- fullname: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- city: string (nullable = true)



In [93]:
nest_list_df1 = spark.createDataFrame(nest_list,ddl_schema)

In [94]:
nest_list_df1.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- fullname: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- city: string (nullable = true)



In [95]:
nest_list_df1.show()

+-----------+----------------+-----+
|customer_id|        fullname| city|
+-----------+----------------+-----+
|          1|{fname1, lname1}|city1|
|          2|{fname2, lname2}|city2|
|          3|{fname3, lname3}|city3|
+-----------+----------------+-----+

